<center>
    <img src="https://raw.githubusercontent.com/dvgodoy/PyTorch101_ODSC_Europe2020/master/images/linear_dogs.jpg" width="900">
</center>

# Yet another math for ML course

# Домашнее задание 3: градиентный спуск и условная оптимизация

В этой домашке две задачи. В первой мы подберём цену билета градиентным спуском: выведем производную руками, напишем спуск, разберёмся с критериями остановки и размером шага. Во второй поделим рекламный бюджет между каналами и выясним, что на самом деле означает множитель Лагранжа.

Всего за домашку можно получить 10 баллов, плюс 0.5 за задание 0 и 0.5 за бонусное задание 1\*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

np.random.seed(42)

## Формат проверки работы: peer review

Каждая работа проверяется несколькими вашими случайно выбранными однокурсниками. Вы также обязаны проверить несколько чужих работ.

Проверяющий выставляет баллы по критериям из задания и оставляет содержательные комментарии: что сделано хорошо, где есть ошибки, что стоит исправить.

Итоговая оценка за задание складывается из трёх частей:

$$
\text{Оценка} = 0.8 \cdot \text{Решение} + 0.1 \cdot \text{Полезность ревью} + 0.1 \cdot \text{Согласованность ревью}
$$

- **80% решение** — это баллы за вашу собственную работу, полученные при проверке.
- **10% полезность ревью** — после проверки авторы работ оценивают, насколько полезными были полученные комментарии. Хорошее ревью должно быть конкретным, вежливым и помогать понять ошибки. Комментарии вида «всё ок», «неверно», «плохо» без объяснений считаются бесполезными.
- **10% согласованность ревью** — оценка проверяющего сравнивается с оценками других проверяющих. Если ваши баллы систематически сильно отличаются от общего консенсуса без убедительных комментариев, эта часть оценки снижается.

Максимум за домашку — 10 баллов, плюс 1 бонусный: 0.5 за задание 0 и 0.5 за задание 1\*. Итого можно набрать 11. Почти под каждым заданием есть вопросы. Ответы на них — часть задания и оцениваются наравне с кодом: работающий код с пустым «Ответ:» полного балла не получает.

Можно:

* указывать на ошибки и спорные места;
* снижать баллы, если решение не соответствует критериям;
* давать рекомендации по исправлению;
* писать развёрнутые комментарии по существу.

Нельзя:

* ставить баллы случайно или всем одинаково;
* писать формальные комментарии без содержания;
* оскорблять автора работы;
* договариваться о взаимных оценках;
* просить других поставить вам определённый балл;
* использовать peer review как способ отомстить или помочь друзьям.

Проверка тоже является частью задания. Халтурные, пустые или недобросовестные ревью будут снижать итоговый балл. Конкретные способы расчёта согласованности и полезности ревью не раскрываются.

__(0.5 балла, бонус) Задание 0: любимый мем__

**ФИО:**

**Ваш любимый мем.** Вставьте его сюда.

> **Для ревьюеров:** полный балл — если мем есть. Если не оч смешно баллы можно снять...

Почти после каждого шага стоит ячейка с проверкой. Она не знает правильного ответа целиком, но знает, каким он обязан быть. Если проверка прошла — идите дальше, если нет — сообщение подскажет, что не так.

In [ ]:
def check(condition, message):
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)

def numeric_derivative(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

# Часть 1: цена билета

Авиакомпания продаёт билеты на рейс. Чем выше цена $p$, тем меньше желающих: спрос линейный,

$$
D(p) = 300 - p
$$

Каждое проданное место обходится компании в $c = 60$. Прибыль с рейса можно найти как:

$$
\Pi(p) = (p - c) \cdot D(p) = (p - 60)(300 - p).
$$

Найдите цену, при которой прибыль компании будет максимальной.

__(1 балл) Задание 1__

- Найдите производную прибыли и вбейте её в функцию ниже.
- Найдите в явном виде оптимальную цену $p^*$.
- Посчитайте $\Pi''(p)$ и убедитесь, что найденное значение — максимум, а не минимум.

Всё нужно проделать руками, либо на бумажке, либо в markdown.

In [ ]:
# впишите свои ответы числами
p_star = ...       # оптимальная цена
Pi_second = ...    # вторая производная (здесь она константа)

In [ ]:
def profit(p):
    return (p - 60) * (300 - p)

def profit_grad(p):
    # YOUR CODE HERE: производная прибыли по цене, формулой
    # ʕ•ᴥ•ʔ
    return ...

In [ ]:
check(abs(profit_grad(100) - numeric_derivative(profit, 100)) < 1e-4, "формула производной совпала с численной")
check(abs(profit_grad(p_star)) < 1e-9, f"в точке p* = {p_star} производная равна нулю")
check(Pi_second < 0, "вторая производная отрицательна: это максимум")

__(1.5 балла) Задание 2: градиентный подъём__

Мы ищем максимум, поэтому будем подниматься в направлении градиента: $p_{t+1} = p_t + \eta\, \Pi'(p_t)$.

Сначала руками: сделайте два шага от $p_0 = 60$ с шагом $\eta = 0.1$ и впишите результат в ячейку ниже.

Затем напишите `gradient_ascent(grad, p0, eta, n_steps)`, которая делает `n_steps` шагов подъёма и возвращает **весь список** точек $p_0, p_1, \ldots, p_{n}$. История нам понадобится для картинок.

In [ ]:
p_1 = ...    # после первого шага от p_0 = 60 с eta = 0.1
p_2 = ...    # после второго шага

In [ ]:
check(abs(p_1 - (60 + 0.1 * profit_grad(60))) < 1e-9, f"первый шаг: p_1 = {p_1}")
check(abs(p_2 - (p_1 + 0.1 * profit_grad(p_1))) < 1e-9, f"второй шаг: p_2 = {p_2}")

In [ ]:
def gradient_ascent(grad, p0, eta, n_steps):
    history = [p0]
    
    # YOUR CODE HERE
    # (⌐■_■)
    
    return np.array(history)

path = gradient_ascent(profit_grad, p0=60, eta=0.1, n_steps=40)
print("первые пять точек:", path[:5])
print("последняя точка:", path[-1])

In [ ]:
check(len(path) == 41, "в истории 41 точка: стартовая плюс 40 шагов")
check(abs(path[1] - p_1) < 1e-9 and abs(path[2] - p_2) < 1e-9, "первые два шага совпали с ручными")
check(abs(path[-1] - p_star) < 0.1, f"за 40 шагов пришли в {path[-1]:.2f}, оптимум {p_star}")

In [ ]:
grid = np.linspace(40, 320, 300)
plt.figure(figsize=(8, 4))
plt.plot(grid, profit(grid), color="gray")
plt.plot(path, profit(path), "o-", color="tab:red", ms=4, lw=1, label="шаги спуска")
plt.axvline(p_star, ls="--", color="gray", lw=1)
plt.xlabel("цена p"); plt.ylabel("прибыль"); plt.legend(); plt.title("Градиентный подъём к оптимальной цене")
plt.show()

Пока спуск делает ровно столько шагов, сколько мы ему сказали. 

В жизни нужен **критерий остановки** — когда прекращать. Три стандартных: 

- функция почти не меняется, $|\Pi(p_{t+1}) - \Pi(p_t)| < \varepsilon$; 
- градиент почти ноль, $|\Pi'(p_t)| < \varepsilon$; 
- точка почти не двигается, $|p_{t+1} - p_t| < \varepsilon$.

Напишите `gradient_ascent_until(grad, f, p0, eta, criterion, eps)`, где `criterion` — строка `"f"`, `"grad"` или `"step"`, и верните историю. Ограничьте число шагов сверху (скажем, 10 000), чтобы не зависнуть, если спуск расходится.

In [ ]:
def gradient_ascent_until(grad, f, p0, eta, criterion="grad", eps=1e-6, max_steps=10_000):
    history = [p0]
    
    # YOUR CODE HERE
    # ┌(ಠ_ಠ)┘
    
    return np.array(history)

for crit in ["f", "grad", "step"]:
    h = gradient_ascent_until(profit_grad, profit, p0=60, eta=0.1, criterion=crit, eps=1e-6)
    print(f"критерий {crit:>4}: {len(h) - 1:4d} шагов, пришли в {h[-1]:.6f}")

In [ ]:
h_f = gradient_ascent_until(profit_grad, profit, 60, 0.1, "f", 1e-6)
h_g = gradient_ascent_until(profit_grad, profit, 60, 0.1, "grad", 1e-6)
h_s = gradient_ascent_until(profit_grad, profit, 60, 0.1, "step", 1e-6)
check(all(abs(h[-1] - p_star) < 1e-2 for h in (h_f, h_g, h_s)), "все три критерия остановились около оптимума")
check(len(h_f) < len(h_g), "критерий по функции остановился раньше, чем по градиенту")
check(len(h_s) < len(h_g), "критерий по шагу остановился раньше, чем по градиенту")

__(1.5 балла) Задание 3: размер шага__

Размер шага $\eta$ -- наш главный параметр. Прогоните спуск от $p_0 = 60$ на 30 шагов с $\eta \in \{0.02,\ 0.1,\ 0.4,\ 0.9,\ 1.05\}$ и нарисуйте все траектории $p_t$ на одном графике (ось $x$ — номер шага).

In [ ]:
etas = [0.02, 0.1, 0.4, 0.9, 1.05]

plt.figure(figsize=(8, 4))

for eta in etas:
    # YOUR CODE HERE
    # (☞ﾟヮﾟ)☞
    pass

plt.axhline(p_star, ls="--", color="gray", lw=1)
plt.ylim(0, 400); plt.xlabel("шаг"); plt.ylabel("цена p"); plt.legend(); plt.title("Траектории при разных η")
plt.show()



На картинке четыре разные траектории. Границы между ними для параболы можно найти руками.

Подставьте $\Pi'(p)$ в шаг $p_{t+1} = p_t + \eta\,\Pi'(p_t)$ и перепишите его в виде

$$
p_{t+1} - p^* = k \cdot (p_t - p^*),
$$

то есть найдите, во сколько раз за один шаг сжимается расстояние до оптимума. Коэффициент $k$ зависит от $\eta$ и от $\Pi''$.

1. При каком $\eta$ коэффициент $k = 0$ — спуск попадает в оптимум **за один шаг** из любой точки?
2. При каком $\eta$ коэффициент $k = -1$ — спуск прыгает через оптимум туда-обратно, не приближаясь? При $\eta$ больше этого он расходится.
3. Выразите оба порога через $\Pi''$, не через конкретные числа.

**Выше решение:**


__(1 балл) Задание 4__

Каждый вопрос — 0.25 балла. Нужен не только ответ, но и объяснение в одно-два предложения.

- В задании 2 критерий остановки по значению функции сработал раньше всех, хотя $\varepsilon$ у всех трёх критериев один. Почему около максимума функция меняется медленнее, чем её производная? 

**Hint:** разложение Тейлора в точке $p^*$.

__Ответ:__


- Критерии «по шагу» и «по градиенту» в этой задаче отличаются ровно в $\eta$ раз. Почему? При каком $\eta$ они сработают на одной и той же итерации?

__Ответ:__

- Для параболы идеальный шаг $1/|\Pi''|$ известен заранее. Почему в реальном обучении им не пользуются напрямую? Что мешает, если функция не парабола?

__Ответ:__

- Пусть спрос стал чувствительнее к цене: $D(p) = 300 - 4p$. Как изменятся идеальный шаг и граница расходимости? Что это говорит о переносе «удачного» $\eta$ с одной задачи на другую?

__Ответ:__

__(0.5 балла, бонус) Задание 5\*: шаг Ньютона__

В задании 3 вы нашли, что для параболы шаг $\eta = 1/|\Pi''|$ приводит в оптимум за одну итерацию. Это частный случай **метода Ньютона**: $p_{t+1} = p_t - \Pi'(p_t) / \Pi''(p_t)$ — шаг подбирается по второй производной в текущей точке.

Примените метод Ньютона к функции, которая уже не парабола: $f(p) = -\big(p^4 - 8p^2 + p\big)$ на отрезке $[-3, 3]$ (у неё два локальных максимума). Запустите из $p_0 = 0.5$ и из $p_0 = -0.5$, выведите траектории, сравните с градиентным подъёмом с $\eta = 0.01$ по числу шагов до $|f'| < 10^{-8}$. Объясните, почему Ньютон сходится так быстро и почему из разных стартов он приходит в разные точки.

In [ ]:
# YOUR CODE HERE
# (❍ᴥ❍ʋ)

# Часть 2: бюджет на рекламу

Авиакомпания запускает рекламу нового направления в двух каналах: контекстная реклама и блогеры. Бюджет $B = 100$. Отдача от канала растёт с бюджетом, но всё медленнее — вторая сотня приносит меньше первой:

$$
R(x, y) = 6\sqrt{x} + 8\sqrt{y},
$$

где $x$ и $y$ — сколько потратили на каждый канал, $R$ — сколько продали билетов, в тысячах. Бюджет надо потратить целиком: $x + y = B$. Нужно поделить его так, чтобы продать больше всего.

В этой части мы решим задачу руками через лагранжиан и градиентным подъёмом, выясним, что означает множитель $\lambda$, научимся делать градиентный подъём по самому лагранжиану.

__(1.25 балла) Задание 6: лагранжиан__

Найдите оптимальное решение через лагранджиан. Всё нужно проделать руками, либо на бумажке, либо в markdown.

In [ ]:
def R(x, y):
    return 6 * np.sqrt(x) + 8 * np.sqrt(y)

B = 100

x_star = ...
y_star = ...
R_star = ...
lam_star = ...

In [ ]:
check(abs(x_star + y_star - B) < 1e-9, "бюджет потрачен целиком")
check(abs(R(x_star, y_star) - R_star) < 1e-9, f"R* = {R_star}")

# перебор по прямой x + y = B: максимум должен быть в найденной точке
xs = np.linspace(0.01, B - 0.01, 100_001)
best = xs[np.argmax(R(xs, B - xs))]
check(abs(best - x_star) < 1e-2, f"перебор по прямой бюджета даёт x = {best:.3f}")

# в оптимуме частные производные R по x и по y равны друг другу и равны λ
dR_dx = numeric_derivative(lambda t: R(t, y_star), x_star)
dR_dy = numeric_derivative(lambda t: R(x_star, t), y_star)
check(abs(dR_dx - lam_star) < 1e-5 and abs(dR_dy - lam_star) < 1e-5, f"∂R/∂x = ∂R/∂y = λ = {lam_star}: предельные отдачи каналов сравнялись")

Последняя проверка — это и есть смысл уравнений $\partial \mathcal L / \partial x = 0$ и $\partial \mathcal L / \partial y = 0$: в оптимуме лишний рубль приносит одинаково в обоих каналах. Если бы в одном канале он приносил больше, рубль стоило бы переложить туда.

__(1 балл) Задание 7: градиентный подъём__

Задачу с одним равенством можно свести к безусловной: подставить $y = B - x$ и максимизировать функцию одной переменной $g(x) = R(x, B - x)$. Посчитайте $g'(x)$ руками, впишите в код и найдите максимум функцией `gradient_ascent_until` из задания 2. Должно получиться $x^*$ из задания 6.

In [ ]:
def g(x):
    return R(x, B - x)

def g_grad(x):
    # YOUR CODE HERE: производная g по x
    # (⌐■_■)
    return ...

path_budget = gradient_ascent_until(g_grad, g, p0=10.0, eta=5.0, criterion="grad", eps=1e-8)
print(f"подъём по прямой бюджета: x = {path_budget[-1]:.4f} за {len(path_budget) - 1} шагов")

In [ ]:
check(abs(g_grad(20.0) - numeric_derivative(g, 20.0)) < 1e-5, "g'(x) совпала с численной производной")
check(abs(path_budget[-1] - x_star) < 1e-3, "градиентный подъём пришёл в x* из задания 6")

plt.figure(figsize=(7, 4))
grid = np.linspace(1, 99, 300)
plt.plot(grid, g(grid), color="gray")
plt.plot(path_budget, g(path_budget), "o-", color="tab:red", ms=3, lw=1, label="шаги подъёма")
plt.axvline(x_star, ls="--", color="gray", lw=1)
plt.xlabel("x — трата на первый канал"); plt.ylabel("R(x, 100 − x)"); plt.legend()
plt.show()

__(1.25 балла) Задание 8: что такое__ $\lambda$

Решите задание 6 ещё раз, но для произвольного бюджета $B$ вместо 100: запишите $x^*(B)$ и максимальную отдачу $R^*(B) = R(x^*(B), y^*(B))$ как функции от $B$. Впишите их в код.

In [ ]:
def x_opt(B):
    # YOUR CODE HERE: оптимальная трата на первый канал при бюджете B
    return ...

def R_opt(B):
    # YOUR CODE HERE: максимальная отдача при бюджете B
    return ...

Посчитайте численно $\dfrac{dR^*}{dB}$ в точке $B = 100$ — на сколько вырастет максимальная отдача, если бюджет увеличить на единицу, — и сравните с $\lambda^*$.

In [ ]:
dR_dB = ...   # YOUR CODE HERE: численная производная R_opt в точке B = 100
print(f"dR*/dB = {dR_dB:.4f},  λ* = {lam_star}")

In [ ]:
check(abs(x_opt(100) - x_star) < 1e-9 and abs(R_opt(100) - R_star) < 1e-9, "при B = 100 формулы дают ответ задания 6")
check(abs(R_opt(49) - R(x_opt(49), 49 - x_opt(49))) < 1e-9, "R*(B) согласуется с x*(B) и при другом B")
check(abs(dR_dB - lam_star) < 1e-4, f"dR*/dB = {dR_dB:.4f} = λ*: множитель Лагранжа — это цена одной единицы бюджета")

**Вопросы (по 0.25):**

- Отдача измеряется в тысячах билетов, бюджет — в деньгах. В каких единицах измеряется $\lambda$? Как бы вы объяснили финансовому директору, что значит $\lambda = 0.5$?

**Ответ:**


- Директор предлагает взять ещё 20 единиц бюджета в кредит, вернуть нужно 28. Билет приносит 1 единицу. Стоит ли? Ответьте точно через $R^*(B)$, а потом объясните, почему прикидка $\lambda \cdot 20$ даёт другой ответ и в какую сторону она ошибается.

**Ответ:**


__(1.5 балла) Задание 9: градиентный подъём по лагранжиану__

В задании 7 мы избавились от ограничения подстановкой $y = B - x$. Это работает, когда ограничение одно и из него легко выразить переменную. В общем случае так не получится, и хочется градиентного метода, который работает прямо с лагранжианом

$$
\mathcal L(x, y, \lambda) = R(x, y) - \lambda\,(x + y - B).
$$

Соблазн — делать градиентный подъём по всем трём переменным сразу. Но так делать нельзя. 

Решение задания 6 — это точка, в которой $\mathcal L$ по $(x, y)$ имеет максимум, а по $\lambda$ — минимум: при фиксированных $x, y$ лагранжиан линеен по $\lambda$, и если $x + y > B$, он неограниченно убывает с ростом $\lambda$. Такая точка называется **седловой**. Подъём по $\lambda$ уведёт от неё, а не к ней.

Правильно будет по $(x, y)$ подниматься, по $\lambda$ спускаться:

$$
x = x + \eta\, \frac{\partial \mathcal L}{\partial x}, \qquad
y = y + \eta\, \frac{\partial \mathcal L}{\partial y}, \qquad
\lambda = \lambda - \eta_\lambda\, \frac{\partial \mathcal L}{\partial \lambda} = \lambda + \eta_\lambda\,(x + y - B).
$$

1. Выпишите все три частные производные $\mathcal L$.
2. Реализуйте схему `lagrangian_ascent_descent` и запустите её из $x_0 = y_0 = 10$, $\lambda_0 = 0$ с $\eta = 5$, $\eta_\lambda = 0.05$ на 2000 шагов. Должны получиться $x^*, y^*, \lambda^*$.
3. Нарисуйте, как ведут себя $x_t + y_t$ и $\lambda_t$ по итерациям.

In [ ]:
def L_grad(x, y, lam):
    
    # YOUR CODE HERE: три частные производные лагранжиана R(x, y) - lam * (x + y - B)
    # ( ͡⊙ ͜ʖ ͡⊙)

    dL_dx = ...
    dL_dy = ...
    dL_dlam = ...
    
    return dL_dx, dL_dy, dL_dlam

def lagrangian_ascent_descent(x, y, lam, eta, eta_lam, n_steps):

    # подъём по x и y, спуск по lam; возвращает историю (x, y, lam) после каждого шага
    history = [(x, y, lam)]
    for _ in range(n_steps):
    
        # YOUR CODE HERE: шаг подъёма по x и y, затем шаг спуска по lam по уже обновлённым x, y
        # (☞ﾟヮﾟ)☞
    
        ...
        history.append((x, y, lam))
    return np.array(history)

hist = lagrangian_ascent_descent(x=10.0, y=10.0, lam=0.0, eta=5.0, eta_lam=0.05, n_steps=2000)
print("пришли в x, y, λ =", hist[-1].round(4))

In [ ]:
dx, dy, dl = L_grad(20.0, 30.0, 0.3)
check(abs(dx - numeric_derivative(lambda t: R(t, 30.0) - 0.3 * (t + 30.0 - B), 20.0)) < 1e-5, "∂L/∂x совпала с численной")
check(abs(dy - numeric_derivative(lambda t: R(20.0, t) - 0.3 * (20.0 + t - B), 30.0)) < 1e-5, "∂L/∂y совпала с численной")
check(abs(dl - (-(20.0 + 30.0 - B))) < 1e-9, "∂L/∂λ = −(x + y − B)")
check(np.allclose(hist[-1], [x_star, y_star, lam_star], atol=1e-2), f"схема сошлась к (x*, y*, λ*) = {hist[-1].round(3)}")
check(abs(hist[-1, 0] + hist[-1, 1] - B) < 1e-2, "бюджет сошёлся: x + y = B")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist[:, 0] + hist[:, 1]); axes[0].axhline(B, ls="--", color="gray", lw=1)
axes[0].set_xlabel("итерация"); axes[0].set_ylabel("x + y"); axes[0].set_title("Траты подстраиваются под бюджет")
axes[1].plot(hist[:, 2], color="tab:red"); axes[1].axhline(lam_star, ls="--", color="gray", lw=1)
axes[1].set_xlabel("итерация"); axes[1].set_ylabel("λ"); axes[1].set_title("Цена рубля сама приходит к λ*")
plt.show()

**Вопросы (по 0.25):**

- Поменяйте в своей функции знак у шага по $\lambda$ — сделайте подъём и по нему тоже — и запустите на 200 шагов. Что произошло с $\lambda_t$ и с $x_t + y_t$?

**Ответ:**


- Кривые на картинках сходятся не монотонно, а колебаниями. Откуда берутся колебания?

**Ответ:**


## Что почитать

* Boyd, Vandenberghe, *Convex Optimization*, глава 5 — откуда берутся множители Лагранжа, почему решение — седловая точка лагранжиана и почему множитель равен производной оптимума по ограничению.
* Николенко, Кадурин, Архангельская, *Глубокое обучение. Погружение в мир нейронных сетей* — глава 2 про градиентный спуск как основу обучения и глава 4 про то, что с ним бывает дальше: momentum, Adagrad, Adam и почему обычный спуск застревает в оврагах.
* [Задачник по глубокому обучению, листок про градиентный спуск](https://fulyankin.github.io/deep_learning_masha_book/problem_set_02_gradient/intro_02.html) — задачи на градиентный спуск, размер шага и его модификации.
* Sebastian Ruder, [An overview of gradient descent optimization algorithms](https://www.ruder.io/optimizing-gradient-descent/) — то же про модификации спуска, коротко и по-английски.